# 🛡️ GigShield — LSTM Disruption Forecaster

**Model:** 3-layer LSTM
**Purpose:** Predict next 7 days disruption probability per Bengaluru zone
**Input:** 90 days of daily features (7 features per day)

This notebook:
1. Generates 3 years of synthetic Bengaluru weather data
2. Creates sliding window sequences
3. Trains the LSTM model
4. Saves model weights
5. Tests forecast for BTM Layout

In [ ]:
!pip install torch numpy pandas -q

## Step 1: Generate 3 Years of Synthetic Weather Data

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

ZONES = ['BTM_LAYOUT', 'KORAMANGALA', 'INDIRANAGAR', 'WHITEFIELD', 'JAYANAGAR',
         'MARATHAHALLI', 'HSR_LAYOUT', 'ELECTRONIC_CITY', 'HEBBAL', 'JP_NAGAR']
LAKE_ZONES = {'HEBBAL', 'HSR_LAYOUT'}

FESTIVAL_DATES = {
    (2023, 11, 12): 'diwali', (2024, 11, 1): 'diwali', (2025, 10, 20): 'diwali',
    (2023, 11, 1): 'kannada', (2024, 11, 1): 'kannada', (2025, 11, 1): 'kannada',
}

def get_season(m):
    if m in [6,7,8,9]: return 'monsoon'
    elif m in [3,4,5]: return 'summer'
    else: return 'winter'

def is_diwali_week(d):
    for (y,m,dy), f in FESTIVAL_DATES.items():
        if f == 'diwali' and abs((d - date(y,m,dy)).days) <= 3: return True
    return False

def is_festival(d):
    for (y,m,dy) in FESTIVAL_DATES:
        if abs((d - date(y,m,dy)).days) <= 1: return True
    return False

np.random.seed(42)
start, end = date(2023,1,1), date(2025,12,31)
days = (end - start).days + 1
records = []

for zone in ZONES:
    lake_bonus = 0.10 if zone in LAKE_ZONES else 0.0
    for i in range(days):
        d = start + timedelta(days=i)
        season = get_season(d.month)
        rain_prob = {'monsoon': 0.45, 'summer': 0.05, 'winter': 0.10}[season] + lake_bonus
        base_aqi = {'monsoon': 85, 'summer': 120, 'winter': 95}[season]
        
        has_rain = np.random.random() < rain_prob
        rainfall = np.random.uniform(8.0, 35.0) if has_rain else np.random.uniform(0.0, 7.5)
        aqi = max(20, min(400, base_aqi + np.random.normal(0, 15) + (80 if is_diwali_week(d) else 0)))
        
        if rainfall >= 8.0: od = np.random.uniform(0.25, 0.60)
        elif aqi >= 200: od = np.random.uniform(0.15, 0.40)
        else: od = np.random.uniform(0.0, 0.15)
        
        disrupted = 1 if (rainfall >= 8.0 and od >= 0.35) or (aqi >= 200 and od >= 0.35) else 0
        
        records.append({
            'zone_id': zone, 'date': d.isoformat(),
            'max_rainfall_mm': round(rainfall, 1), 'mean_aqi': round(aqi),
            'order_drop_pct': round(od, 4), 'day_of_week': d.weekday(),
            'month': d.month, 'is_festival': int(is_festival(d)),
            'historical_claim_count': np.random.poisson(disrupted * 3 + 0.5),
            'disrupted': disrupted,
        })

df = pd.DataFrame(records)
print(f'Generated {len(df)} records for {len(ZONES)} zones over {days} days')
print(f'Disruption rate: {df["disrupted"].mean():.2%}')
df.head()

## Step 2: Define LSTM Model

In [ ]:
import torch
import torch.nn as nn

class DisruptionForecaster(nn.Module):
    def __init__(self, input_size=7, hidden_size=128, num_layers=3, output_days=7, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                           num_layers=num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_days)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        out = self.fc(last_hidden)
        return self.sigmoid(out)

print('LSTM model defined ✅')

## Step 3: Prepare Sliding Window Sequences

In [ ]:
SEQ_LEN = 90
OUTPUT_DAYS = 7
FEATURES = ['max_rainfall_mm', 'mean_aqi', 'order_drop_pct', 'day_of_week', 'month', 'is_festival', 'historical_claim_count']

def prepare_sequences(df, zone_id):
    zd = df[df['zone_id'] == zone_id].sort_values('date').reset_index(drop=True)
    vals = zd[FEATURES].values.astype(np.float32)
    targets = zd['disrupted'].values.astype(np.float32)
    X, y = [], []
    for i in range(len(vals) - SEQ_LEN - OUTPUT_DAYS + 1):
        X.append(vals[i:i+SEQ_LEN])
        y.append(targets[i+SEQ_LEN:i+SEQ_LEN+OUTPUT_DAYS])
    return np.array(X), np.array(y)

all_X, all_y = [], []
for zone in ZONES:
    X, y = prepare_sequences(df, zone)
    all_X.append(X)
    all_y.append(y)
    print(f'{zone}: {len(X)} sequences')

X_train = np.concatenate(all_X)
y_train = np.concatenate(all_y)
print(f'\nTotal: {len(X_train)} sequences, shape: {X_train.shape}')

## Step 4: Train the LSTM

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = DisruptionForecaster().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    total_loss = 0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        avg = total_loss / len(loader)
        print(f'Epoch {epoch+1}/100 — Loss: {avg:.4f}')

torch.save(model.state_dict(), 'forecast_model.pt')
print('\n✅ Model saved to forecast_model.pt')

## Step 5: Test Forecast for BTM Layout

In [ ]:
model.eval()
# Use last 90 days of BTM Layout data for prediction
btm = df[df['zone_id'] == 'BTM_LAYOUT'].sort_values('date').tail(90)
test_input = btm[FEATURES].values.astype(np.float32)
test_tensor = torch.from_numpy(test_input).unsqueeze(0).to(device)

with torch.no_grad():
    probs = model(test_tensor)[0].cpu().numpy()

print('\n📊 BTM Layout — Next 7 Days Disruption Forecast:')
for i, p in enumerate(probs):
    risk = 'HIGH 🔴' if p >= 0.60 else ('MEDIUM 🟡' if p >= 0.35 else 'LOW 🟢')
    bar = '█' * int(p * 40)
    print(f'  Day {i+1}: {p:.2%} {risk} {bar}')

## Step 6: Download Model

Download `forecast_model.pt` and place it in `backend/ml/forecast/`

In [ ]:
try:
    from google.colab import files
    files.download('forecast_model.pt')
except:
    print('Not in Colab. File saved locally.')